In [1]:
import tqdm as notebook_tqdm
from sentence_transformers import SentenceTransformer

/home/saquib-siddiqui/tensorvault/learning/cb-ai/code_files/cb-ai-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Always run `docker run -p 6333:6333 -p 6334:6334 qdrant/qdrant` in terminal to start qdrant

In [2]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3053.83it/s]


In [6]:
# Example Documents
docs = ["Dogs are loyal and friendly domestic animals.",
        "Cats are independent and curious creatures.", 
        "The Milky Way galaxy contains over 200 billion stars.",
        "London is the capital city of England and the United Kingdom.",
        ]

In [7]:
more_docs = ["The Great Wall of China is a historic fortification built to protect against invasions.",
            "The Amazon Rainforest is the largest tropical rainforest in the world, known for its biodiversity.",
            "The Eiffel Tower is an iconic landmark in Paris, France, and a symbol of architectural innovation.",
            "The Sahara Desert is the largest hot desert in the world, covering much of North Africa.",
            "The Taj Mahal is a UNESCO World Heritage Site and a symbol of love and architectural beauty in India.",
            "The Grand Canyon is a natural wonder in the United States, known for its breathtaking landscapes and geological formations.",
            "The Great Barrier Reef is the world's largest coral reef system, located off the coast of Australia and home to diverse marine life.",
            "The Pyramids of Giza in Egypt are ancient structures that have fascinated historians and archaeologists for centuries.",
            "Cats are curious and agile animals, often displaying playful behavior and independent personalities."]
docs.extend(more_docs)

In [8]:
docs

['Dogs are loyal and friendly domestic animals.',
 'Cats are independent and curious creatures.',
 'The Milky Way galaxy contains over 200 billion stars.',
 'London is the capital city of England and the United Kingdom.',
 'The Great Wall of China is a historic fortification built to protect against invasions.',
 'The Amazon Rainforest is the largest tropical rainforest in the world, known for its biodiversity.',
 'The Eiffel Tower is an iconic landmark in Paris, France, and a symbol of architectural innovation.',
 'The Sahara Desert is the largest hot desert in the world, covering much of North Africa.',
 'The Taj Mahal is a UNESCO World Heritage Site and a symbol of love and architectural beauty in India.',
 'The Grand Canyon is a natural wonder in the United States, known for its breathtaking landscapes and geological formations.',
 "The Great Barrier Reef is the world's largest coral reef system, located off the coast of Australia and home to diverse marine life.",
 'The Pyramids o

In [9]:
# Creating Embedding
embedding = embedder.encode(docs)
embedding.shape

(13, 384)

In [10]:
custom_query = "What is the capital of England?"
query_embedding = embedder.encode(custom_query)

In [11]:
scores = embedding @ query_embedding.T
scores.shape

(13,)

In [12]:
scores

array([ 0.03267261, -0.00313605,  0.02374041,  0.6945894 ,  0.05492312,
        0.06121574,  0.13497373,  0.03239074,  0.0510678 ,  0.03690982,
        0.06854003, -0.04782426, -0.03716575], dtype=float32)

In [13]:
scores = embedder.similarity(query_embedding, embedding)

In [14]:
scores

tensor([[ 0.0327, -0.0031,  0.0237,  0.6946,  0.0549,  0.0612,  0.1350,  0.0324,
          0.0511,  0.0369,  0.0685, -0.0478, -0.0372]])

## Vector Databses

In [15]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)

In [16]:
# Connecting to a local instance
client = QdrantClient(path="http://localhost:6333")

In [17]:
# Creating a Collection

COLLECTION_NAME = "docs"
DIM = embedder.get_embedding_dimension()

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=DIM,
        distance=Distance.COSINE, # try using what model was trained on, e.g. COSINE for MiniLM
    ),
)
print("Collection created.")

Collection created.


In [18]:
for idx, doc in enumerate(docs):
    embedding = embedder.encode(doc)
    point = PointStruct(
        id=idx,
        vector=embedding.tolist(),
        payload={"text": doc},
    )
    print(f"Upserting point {idx} with embedding dimension {len(embedding)}")
    # print(f"Embedding: {embedding}")
    print(f"Payload: {point.payload}")
    print(f"Point ID: {point.id}")
    print(f"Point Vector: {point.vector}")
    print(f"Point: {point}")

    response = client.upsert(
        collection_name=COLLECTION_NAME,
        points=[point],
        wait=True,
    )
print(response.status)

Upserting point 0 with embedding dimension 384
Payload: {'text': 'Dogs are loyal and friendly domestic animals.'}
Point ID: 0
Point Vector: [-0.036633919924497604, -0.03294789046049118, 0.022733749821782112, 0.06643646210432053, -0.05361654609441757, 0.02274583838880062, -0.021171478554606438, -0.08273489773273468, 0.032235078513622284, 0.049956027418375015, 0.05676425248384476, -0.05348464101552963, 0.03142577409744263, 0.04759388789534569, 0.03200981393456459, 0.02323566935956478, -0.02294355258345604, -0.04777488484978676, -0.020906822755932808, -0.0023211208172142506, -0.13793084025382996, -0.02339058741927147, 0.01779065653681755, 0.0007324256584979594, -0.06379149854183197, 0.016908591613173485, 0.05821601301431656, -0.05056997388601303, 0.020162250846624374, -0.008607838302850723, -0.026648318395018578, -0.036464933305978775, 0.042313262820243835, 0.018132474273443222, -0.0742080882191658, 0.0009708097786642611, -0.005285808350890875, 0.027800675481557846, 0.016398563981056213, 

In [19]:
# query_demo

custom_query = "What is the capital of England?"
query_embedding = embedder.encode(custom_query)
query_vector = query_embedding.tolist()

In [20]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
    #score_threshold=0.30
)

for r in results.points:
    print(f"Score: {r.score:.4f} | {r.payload['text']}")

Score: 0.6946 | London is the capital city of England and the United Kingdom.
Score: 0.1350 | The Eiffel Tower is an iconic landmark in Paris, France, and a symbol of architectural innovation.
Score: 0.0685 | The Great Barrier Reef is the world's largest coral reef system, located off the coast of Australia and home to diverse marine life.


In [21]:
custom_queries = ["What is the capital of England?", 
                  "What is the largest desert in the world?", 
                  "What is the Great Wall of China?", 
                  "how are cats"]

In [22]:
for query in custom_queries:
    query_embedding = embedder.encode(query)
    query_vector = query_embedding.tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=1,
        #score_threshold=0.30
    )

    print(f"\nQuery: {query}")
    for r in results.points:
        print(f"Score: {r.score:.4f} | {r.payload['text']}")


Query: What is the capital of England?
Score: 0.6946 | London is the capital city of England and the United Kingdom.

Query: What is the largest desert in the world?
Score: 0.6570 | The Sahara Desert is the largest hot desert in the world, covering much of North Africa.

Query: What is the Great Wall of China?
Score: 0.6812 | The Great Wall of China is a historic fortification built to protect against invasions.

Query: how are cats
Score: 0.6208 | Cats are independent and curious creatures.
